# 02 · Dim_Producto
**IDY1102 · EP3 · DataRetail 360**

| Campo | Detalle |
|---|---|
| Responsable | [Nombre del integrante] |
| Tabla destino | `dim_producto` |
| SCD | **Tipo 2**: marca, categoría, subcategoría, línea de negocio · Tipo 1: nombre del producto |
| Clave natural | `sku` |
| Depende de | — |
| Fuentes | `productos_snapshot_1.csv` y `productos_snapshot_2.csv` |
| Alimenta a | `fact_ventas` (notebook 04) |

> **Cómo usar esta plantilla.** Las celdas marcadas **✏️ TODO** son la parte que debe implementar el integrante
> responsable (las reglas de calidad). Todo lo demás ya está conectado a los módulos compartidos de `src/`
> (carga idempotente, SCD, auditoría, logging). Antes de commitear: *Restart & Run All* debe terminar sin errores.

## 0. Setup

In [ ]:
import os, sys
from datetime import date
from pathlib import Path

import pandas as pd

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))

from src import db, extract, medallion, quality as q, schema
from src.config import get_settings
from src.logger import get_logger

PROCESO = "dim_producto"
log = get_logger(PROCESO)
cfg = get_settings()
engine = db.get_engine(cfg)
schema.crear_esquema(engine)                  # idempotente: crea solo lo que falta
schema.sembrar_miembros_desconocidos(engine)  # miembro -1 "No informado" en cada dimensión
pd.set_option("display.width", 160)

## 1. Parámetros

In [ ]:
# Snapshot a procesar: lo define run_pipeline.py (variable de entorno SNAPSHOT) o puedes cambiarlo a mano.
SNAPSHOT = int(os.getenv("SNAPSHOT", "1"))
FECHAS_EFECTIVAS = {1: date(2025, 1, 1), 2: date(2025, 6, 1)}   # desde cuándo rige cada snapshot
FECHA_EFECTIVA = FECHAS_EFECTIVAS[SNAPSHOT]
ARCHIVO = cfg.data_dir / f"productos_snapshot_{SNAPSHOT}.csv"
print(f"Snapshot {SNAPSHOT} · fecha efectiva {FECHA_EFECTIVA} · {ARCHIVO.name}")

## 2. Extract
Lee la fuente cruda y la guarda **sin cambios** en la capa Bronze.

In [ ]:
def extraer(ruta):
    """Lee la fuente cruda y la deja intacta en la capa Bronze."""
    df = extract.leer_csv(ruta)
    medallion.guardar_capa(df, "bronze", f"productos_snapshot_{SNAPSHOT}")
    return df

## 3. Transform  ✏️ TODO
**Implementa al menos 3 reglas de calidad** (la rúbrica del EP3 y la Actividad 2.2 lo piden). Estas son las que traen los datos de ejemplo:

| Problema en los datos de ejemplo | Dimensión de calidad | Pista |
|---|---|---|
| SKU con minúsculas y espacios (`' tv-004 '`) | Consistencia | `q.normalizar_sku` |
| Mismo producto dos veces con SKU escrito distinto (`cf-005` / `CF-005`) | Unicidad | normaliza primero y recién después busca duplicados |
| Fila sin SKU | Completitud | máscara `isna() | == ''` y `q.separar_rechazos` |
| Textos descriptivos con espacios sobrantes | Consistencia | `q.normalizar_texto` |

In [ ]:
# Contrato de salida de transformar(): columnas que espera la carga (no lo borres)
COLUMNAS_SALIDA = ["sku", "nombre_producto", "marca", "categoria", "subcategoria", "linea_negocio"]

In [ ]:
def transformar(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Limpia y valida. Retorna (df_limpio, df_rechazos).

    ✏️ TODO (integrante responsable). Pasos sugeridos:
      1. Homologar el SKU (`q.normalizar_sku`) ANTES de buscar duplicados.
      2. Rechazar SKU vacíos y duplicados (conservando la primera aparición) con `q.separar_rechazos`.
      3. Normalizar los atributos descriptivos.
      4. Devolver `df_limpio[COLUMNAS_SALIDA]` (una fila por SKU) y los rechazos consolidados.
    """
    raise NotImplementedError("TODO: implementar transformar() de dim_producto")

## 4. Load
Carga idempotente con el helper compartido y ejecución completa con auditoría (`Ejecucion`).

In [ ]:
ATRIBUTOS_SCD2 = ["marca", "categoria", "subcategoria", "linea_negocio"]
ATRIBUTOS_SCD1 = ["nombre_producto"]

def cargar(df_limpio: pd.DataFrame, df_rechazos: pd.DataFrame) -> dict:
    """Registra los rechazos y carga la dimensión aplicando SCD Tipo 2."""
    assert set(COLUMNAS_SALIDA) <= set(df_limpio.columns), "transformar() no devolvió las columnas esperadas"
    db.registrar_rechazos(engine, PROCESO, df_rechazos)
    return db.aplicar_scd2(engine, "dim_producto", df_limpio, clave_natural="sku",
                           atributos_scd2=ATRIBUTOS_SCD2, fecha_efectiva=FECHA_EFECTIVA,
                           key_col="producto_key", atributos_scd1=ATRIBUTOS_SCD1)

In [ ]:
with db.Ejecucion(engine, PROCESO) as run:
    df_crudo = extraer(ARCHIVO)
    run.leidas = len(df_crudo)
    df_limpio, df_rechazos = transformar(df_crudo)
    medallion.guardar_capa(df_limpio, "silver", f"productos_snapshot_{SNAPSHOT}")
    resumen = cargar(df_limpio, df_rechazos)
    run.cargadas = resumen["nuevos"] + resumen["versiones_nuevas"]
    run.rechazadas = len(df_rechazos)
resumen

## 5. Validación
Chequeos automáticos sobre lo ya cargado. Si algo está mal, el notebook **falla** (no sigue en silencio).

In [ ]:
def validar():
    dim = db.leer_tabla(engine, "dim_producto")
    problemas = q.chequear_scd2(dim, "sku", "producto_key")
    assert not problemas, problemas
    assert (dim["producto_key"] == -1).sum() == 1, "debe existir exactamente un miembro 'No informado' (-1)"
    log.info("Validación de dim_producto OK (%s filas)", len(dim))

validar()

In [ ]:
# Idempotencia: volver a cargar los MISMOS datos no debe cambiar nada
filas_antes = len(db.leer_tabla(engine, "dim_producto"))
resumen_2 = cargar(df_limpio, pd.DataFrame(columns=q.COLUMNAS_RECHAZO))
assert len(db.leer_tabla(engine, "dim_producto")) == filas_antes, "la carga no es idempotente"
print("Idempotencia OK:", resumen_2)

## 6. Evidencia
Estas salidas son las que se capturan para el informe (apartados 6, 7 y 9 de la pauta EP3).

In [ ]:
def conteos_por_capa() -> pd.DataFrame:
    """Filas en cada capa Medallion para este snapshot: trazabilidad fuente → bronze → silver → gold."""
    bronze = medallion.leer_capa("bronze", f"productos_snapshot_{SNAPSHOT}")
    silver = medallion.leer_capa("silver", f"productos_snapshot_{SNAPSHOT}")
    gold = db.leer_tabla(engine, "dim_producto", solo_vigentes=True)
    gold = gold[gold["producto_key"] != -1]
    return pd.DataFrame({"capa": ["bronze (crudo)", "silver (limpio)", "gold (vigentes en el DW)"],
                          "filas": [len(bronze), len(silver), len(gold)]})

conteos_por_capa()

In [ ]:
# Historia SCD2 de un registro de ejemplo (AU-002, que pasa de Accesorios a Audio en el snapshot 2): una fila por versión
dim = db.leer_tabla(engine, "dim_producto")
versiones = dim.groupby("sku").size()
ejemplos = versiones[versiones > 1].index[:1]
dim[dim["sku"].isin(ejemplos)].sort_values(["sku", "valido_desde"])

In [ ]:
# Rechazos de este proceso y últimas ejecuciones registradas (auditoría)
rech = db.leer_tabla(engine, "auditoria_rechazos")
display(rech[rech["proceso"] == PROCESO].groupby("motivo_rechazo").size().rename("filas").to_frame())
ejec = db.leer_tabla(engine, "auditoria_ejecuciones")
ejec[ejec["proceso"] == PROCESO].tail(4)[["inicio", "estado", "filas_leidas", "filas_cargadas", "filas_rechazadas"]]

## 7. Decisiones de diseño (completar)
- El profesor pidió evaluar costo-beneficio entre SCD 1, 2 y 3. ¿Qué atributos de producto cambian seguido y cuáles no? ¿Alguno justificaría Tipo 3 (guardar solo el valor anterior)?
- ¿Qué consulta del negocio se rompería si `categoria` fuera Tipo 1? (ej. venta por categoría de marzo).
- Jerarquía: SKU → subcategoría → categoría → línea de negocio.